In [1]:
# Step 1: Import required libraries and modules
import sys, os
sys.path.append(os.path.abspath("..")) 

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from models.mobilenetv3 import MobileNetV3Extractor
from models.lstm_attention import BiLSTMWithAttention
from preprocessing.dataset import SignLanguageDataset


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: False


In [2]:
# Step 2: Define image transforms and label mapping 

train_transform = A.Compose([
    A.Resize(128, 128),  
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()  # PyTorch tensor
])

train_root = "../data/frames/train"

# Obtain the category list and establish the tag mapping.
class_names = sorted([
    d for d in os.listdir(train_root)
    if os.path.isdir(os.path.join(train_root, d)) and not d.startswith(".")
])
label_map = {name: idx for idx, name in enumerate(class_names)}

print("Discovered classes:", label_map)


Discovered classes: {'accident': 0, 'africa': 1, 'all': 2, 'apple': 3, 'basketball': 4, 'bed': 5, 'before': 6, 'bird': 7, 'birthday': 8, 'black': 9, 'blue': 10, 'book': 11, 'bowling': 12, 'brown': 13, 'but': 14, 'can': 15, 'candy': 16, 'chair': 17, 'change': 18, 'cheat': 19, 'city': 20, 'clothes': 21, 'color': 22, 'computer': 23, 'cook': 24, 'cool': 25, 'corn': 26, 'cousin': 27, 'cow': 28, 'dance': 29, 'dark': 30, 'deaf': 31, 'decide': 32, 'doctor': 33, 'dog': 34, 'drink': 35, 'eat': 36, 'enjoy': 37, 'family': 38, 'fine': 39, 'finish': 40, 'fish': 41, 'forget': 42, 'full': 43, 'give': 44, 'go': 45, 'graduate': 46, 'hat': 47, 'hearing': 48, 'help': 49, 'hot': 50, 'how': 51, 'jacket': 52, 'kiss': 53, 'language': 54, 'last': 55, 'later': 56, 'letter': 57, 'like': 58, 'man': 59, 'many': 60, 'medicine': 61, 'meet': 62, 'mother': 63, 'need': 64, 'no': 65, 'now': 66, 'orange': 67, 'paint': 68, 'paper': 69, 'pink': 70, 'pizza': 71, 'play': 72, 'pull': 73, 'purple': 74, 'right': 75, 'same': 76,

In [3]:
# Step 3: Build dataset and dataloader
train_dataset = SignLanguageDataset(
    root_dir=train_root,
    label_map=label_map,
    transform=train_transform
)

# 添加标签分布检查代码
from collections import Counter
labels = [label for _, label in train_dataset.samples]
print("标签分布：", Counter(labels))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)



[DEBUG] First 5 samples loaded:
Sample 0: class=apple, label=3
    Frame count: 16 | First frame: ../data/frames/train/apple/68003/frame_000.jpg
Sample 1: class=apple, label=3
    Frame count: 16 | First frame: ../data/frames/train/apple/65084/frame_000.jpg
Sample 2: class=apple, label=3
    Frame count: 16 | First frame: ../data/frames/train/apple/70309/frame_000.jpg
Sample 3: class=apple, label=3
    Frame count: 16 | First frame: ../data/frames/train/apple/03003/frame_000.jpg
Sample 4: class=apple, label=3
    Frame count: 16 | First frame: ../data/frames/train/apple/65085/frame_000.jpg
标签分布： Counter({6: 16, 35: 15, 63: 14, 23: 14, 5: 13, 45: 13, 87: 13, 58: 12, 82: 12, 12: 12, 38: 12, 25: 12, 31: 12, 86: 12, 97: 12, 93: 12, 16: 12, 67: 12, 3: 11, 69: 11, 59: 11, 48: 11, 34: 11, 65: 11, 78: 11, 22: 11, 80: 11, 70: 11, 66: 11, 27: 11, 94: 11, 29: 11, 50: 11, 56: 11, 99: 11, 71: 11, 9: 11, 79: 11, 95: 11, 84: 11, 0: 10, 18: 10, 60: 10, 10: 10, 62: 10, 73: 10, 46: 10, 54: 10, 7: 10, 2

In [ ]:
# Step 4: Define full model structure

class FullSLRModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.feature_extractor = MobileNetV3Extractor()
        self.temporal_model = BiLSTMWithAttention(
            input_dim=960, hidden_dim=128, num_classes=100, num_layers=2)

    def forward(self, x):  # x: [B, T, C, H, W]
        
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.feature_extractor(x)  # [B*T, 960]
        features = features.view(B, T, -1)    # [B, T, 960]
        logits, _ = self.temporal_model(features)
        return logits

In [ ]:
#  Step 5: Set device, initialize model, loss function, optimizer 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FullSLRModel(num_classes=len(label_map)).to(device)

# Use cross-entropy loss with label smoothing.
criterion = nn.CrossEntropyLoss(label_smoothing=0)

# Adam optimizer + weight decay
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Train with automatic mixed precision
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()


/opt/anaconda3/envs/slr-env/lib/python3.9/site-packages/torch/cuda/amp/grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [ ]:
# Step 6: Setup validation dataloader


val_root = "../data/frames/validation"  

val_transform = A.Compose([
    # A.Resize(128, 128),
    # A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_dataset = SignLanguageDataset(
    root_dir=val_root,
    label_map=label_map,
    transform=val_transform
)

val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)



[DEBUG] First 5 samples loaded:
Sample 0: class=apple, label=3
    Frame count: 20 | First frame: ../data/frames/validation/apple/69213/frame_000.jpg
Sample 1: class=apple, label=3
    Frame count: 20 | First frame: ../data/frames/validation/apple/02999/frame_000.jpg
Sample 2: class=apple, label=3
    Frame count: 20 | First frame: ../data/frames/validation/apple/65086/frame_000.jpg
Sample 3: class=accident, label=0
    Frame count: 20 | First frame: ../data/frames/validation/accident/00626/frame_000.jpg
Sample 4: class=accident, label=0
    Frame count: 20 | First frame: ../data/frames/validation/accident/00627/frame_000.jpg


In [ ]:

# Step 7 Define validation function 

def evaluate(model, val_loader, criterion):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = val_loss / len(val_loader)
    accuracy = correct / total
    return avg_loss, accuracy


In [ ]:
# Step 8
# Initialize the record list
train_loss_list, val_loss_list = [], []
train_acc_list, val_acc_list = [], []
best_val_acc = 0.0

# Define the verification function
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for videos, labels in tqdm(val_loader, desc='Validating'):
            videos = videos.to(device)
            labels = labels.to(device)
            
            with autocast():
                outputs = model(videos)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = 100. * correct / total
    return val_loss, val_acc

# training loop
num_epochs = 10  
for epoch in range(num_epochs):
    # training stage
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for batch_idx, (videos, labels) in enumerate(loop):
        videos = videos.to(device)
        labels = labels.to(device)
        
        # Gradient zeroing
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(videos)
            loss = criterion(outputs, labels)
        
        # Backpropagation and Optimization
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Calculation accuracy
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        train_loss += loss.item()
        
        # Update progress bar
        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)
    
    # Calculate and save the training metrics.
    train_loss = train_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)
    
    # verification stage
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)
    
    # Save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'New best model saved with val_acc: {val_acc:.2f}%')
    
    # Print the results of the current epoch.
    print(f'Epoch: {epoch+1}/{num_epochs} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')



Epoch 1/2:   0%|          | 0/622 [00:00<?, ?it/s]

/opt/anaconda3/envs/slr-env/lib/python3.9/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Epoch 1/2:  52%|█████▏    | 323/622 [45:11<36:04,  7.24s/it, acc=0.232, loss=5.71]  

In [ ]:
# Step 9
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(train_loss_list, label='Train Loss')
plt.plot(val_loss_list, label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# accuracy curve
plt.subplot(1, 2, 2)
plt.plot(train_acc_list, label='Train Acc')
plt.plot(val_acc_list, label='Val Acc')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

print(f'Best Validation Accuracy: {best_val_acc:.2f}%')